In [ ]:
"""
Outfitly - H&M Dataset Data Preparation Script
================================================
Maps H&M parquet data files to the Outfitly SQL Server schema:
  - articles.parquet  → dbo.Products
  - customers.parquet → dbo.AspNetUsers
  - transactions_train.parquet → dbo.UserInteractions (new table)

Also exports style_tags.csv for model training (Feature Engineering).

Usage:
  python data_preparation.py              # Full run: load → clean → insert into SQL Server
  python data_preparation.py --dry-run    # Load & preview only, no SQL writes
  python data_preparation.py --verify     # Query DB and print summary counts
"""

In [ ]:
import argparse
import os
import sys
import uuid
from datetime import datetime

In [ ]:
import pandas as pd
from sqlalchemy import (
    Column,
    DateTime,
    Float,
    Integer,
    MetaData,
    String,
    Table,
    create_engine,
    inspect,
    text,
)

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
SCRIPT_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "wwwroot")  # parquet files live here
OUTPUT_DIR = os.path.join(SCRIPT_DIR, "data")      # derived CSVs for model training

In [ ]:
# SQL Server connection string (matches appsettings.json – LocalDB)
# pyodbc uses the ODBC driver; most Windows machines have the ODBC 17/18 driver
CONNECTION_STRING = (
    "mssql+pyodbc://@(localdb)\\mssqllocaldb/Outfitly"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&TrustServerCertificate=yes"
)

In [ ]:
# How many rows to insert per batch (to avoid memory issues with 31M transactions)
BATCH_SIZE = 10_000

In [ ]:
# Max transactions to process (None = all). Set to a number for quick testing.
MAX_TRANSACTIONS = None

---------------------------------------------------------------------------
Helpers
---------------------------------------------------------------------------

In [ ]:
def load_parquet(name: str) -> pd.DataFrame:
    """Load a parquet file from the DATA_DIR."""
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print(f"ERROR: File not found: {path}")
        sys.exit(1)
    print(f"Loading {name} ...")
    df = pd.read_parquet(path, engine='fastparquet')
    print(f"  → {df.shape[0]:,} rows × {df.shape[1]} columns")
    return df

In [ ]:
def ensure_output_dir():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

---------------------------------------------------------------------------
Step 1-A: Articles → Products
---------------------------------------------------------------------------

In [ ]:
def prepare_products(df_articles: pd.DataFrame) -> pd.DataFrame:
    """
    Map H&M articles to the Outfitly Products table schema.

    Outfitly Products columns:
        Id (int), Name, Price, ImageUrls, Description, Category,
        AvailableColors, CreatedAt
    """
    df = df_articles.copy()

    products = pd.DataFrame()
    products["Id"] = df["article_id"].astype(int)
    products["Name"] = df["prod_name"].fillna("Unknown")
    products["Description"] = df["detail_desc"].fillna("")
    products["Category"] = df["product_type_name"].fillna("Unknown")
    # H&M prices are normalised fractions; multiply by 1000 to get reasonable $
    products["Price"] = 29.99  # Default placeholder price
    products["ImageUrls"] = ""  # No images in H&M dataset
    products["AvailableColors"] = ""  # We use style_tags separately
    products["CreatedAt"] = datetime.now()

    # De-dup: keep first occurrence per article_id (should already be unique)
    products = products.drop_duplicates(subset=["Id"], keep="first")

    print(f"  Products prepared: {len(products):,} unique items")
    return products

---------------------------------------------------------------------------
Step 1-B: Customers → AspNetUsers
---------------------------------------------------------------------------

In [ ]:
def prepare_users(df_customers: pd.DataFrame) -> pd.DataFrame:
    """
    Map H&M customers to ASP.NET Identity AspNetUsers table.

    Required columns for Identity:
        Id (nvarchar 450), UserName, NormalizedUserName, Email,
        NormalizedEmail, EmailConfirmed, PasswordHash,
        SecurityStamp, ConcurrencyStamp, PhoneNumberConfirmed,
        TwoFactorEnabled, LockoutEnabled, AccessFailedCount
    """
    df = df_customers.copy()

    users = pd.DataFrame()
    users["Id"] = df["customer_id"].astype(str)
    users["UserName"] = df["customer_id"].astype(str).str[:20]  # Truncate for display
    users["NormalizedUserName"] = users["UserName"].str.upper()
    users["Email"] = users["Id"].str[:10] + "@hm-import.local"
    users["NormalizedEmail"] = users["Email"].str.upper()
    users["EmailConfirmed"] = False
    users["PasswordHash"] = None
    users["SecurityStamp"] = [str(uuid.uuid4()) for _ in range(len(users))]
    users["ConcurrencyStamp"] = [str(uuid.uuid4()) for _ in range(len(users))]
    users["PhoneNumber"] = None
    users["PhoneNumberConfirmed"] = False
    users["TwoFactorEnabled"] = False
    users["LockoutEnd"] = None
    users["LockoutEnabled"] = True
    users["AccessFailedCount"] = 0

    users = users.drop_duplicates(subset=["Id"], keep="first")
    print(f"  Users prepared: {len(users):,} unique customers")
    return users

---------------------------------------------------------------------------
Step 1-C: Transactions → UserInteractions
---------------------------------------------------------------------------

In [ ]:
def prepare_interactions(df_txn: pd.DataFrame) -> pd.DataFrame:
    """
    Map H&M transactions to a new UserInteractions table.

    Schema: Id (identity), UserId, ProductId, InteractionType, Price, Timestamp
    """
    df = df_txn.copy()

    if MAX_TRANSACTIONS is not None:
        df = df.head(MAX_TRANSACTIONS)
        print(f"  (Limited to first {MAX_TRANSACTIONS:,} transactions)")

    interactions = pd.DataFrame()
    interactions["UserId"] = df["customer_id"].astype(str)
    interactions["ProductId"] = df["article_id"].astype(int)
    interactions["InteractionType"] = "Purchase"
    interactions["Price"] = df["price"].astype(float)
    interactions["Timestamp"] = pd.to_datetime(df["t_dat"])

    print(f"  Interactions prepared: {len(interactions):,} records")
    return interactions

---------------------------------------------------------------------------
Step 1-D: Export Style Tags for Model Training
---------------------------------------------------------------------------

In [ ]:
def export_style_tags(df_articles: pd.DataFrame):
    """Save colour_group_name and graphical_appearance_name for feature engineering."""
    ensure_output_dir()

    tags = df_articles[["article_id", "colour_group_name", "graphical_appearance_name",
                         "product_group_name", "index_group_name",
                         "section_name", "garment_group_name"]].copy()
    tags = tags.rename(columns={"article_id": "Id"})
    tags["Id"] = tags["Id"].astype(int)

    out_path = os.path.join(OUTPUT_DIR, "style_tags.csv")
    tags.to_csv(out_path, index=False)
    print(f"  Style tags exported → {out_path} ({len(tags):,} rows)")

---------------------------------------------------------------------------
SQL Insert Logic
---------------------------------------------------------------------------

In [ ]:
def get_engine():
    return create_engine(CONNECTION_STRING, fast_executemany=True)

In [ ]:
def create_user_interactions_table(engine):
    """Create UserInteractions table if it doesn't exist."""
    metadata = MetaData()
    Table(
        "UserInteractions",
        metadata,
        Column("Id", Integer, primary_key=True, autoincrement=True),
        Column("UserId", String(450), nullable=False),
        Column("ProductId", Integer, nullable=False),
        Column("InteractionType", String(50), nullable=False),
        Column("Price", Float, nullable=True),
        Column("Timestamp", DateTime, nullable=True),
    )
    metadata.create_all(engine)
    print("  UserInteractions table ensured.")

In [ ]:
def insert_products(engine, df_products: pd.DataFrame):
    """Insert products, skipping any IDs that already exist."""
    insp = inspect(engine)
    if not insp.has_table("Products"):
        print("  WARNING: Products table does not exist. Skipping product insert.")
        return

    # Get existing product IDs to avoid duplicates
    with engine.connect() as conn:
        existing = pd.read_sql("SELECT Id FROM Products", conn)
    existing_ids = set(existing["Id"].values)

    new_products = df_products[~df_products["Id"].isin(existing_ids)]
    if len(new_products) == 0:
        print("  No new products to insert (all IDs already exist).")
        return

    print(f"  Inserting {len(new_products):,} new products ...")
    total = 0
    for start in range(0, len(new_products), BATCH_SIZE):
        batch = new_products.iloc[start : start + BATCH_SIZE]
        with engine.begin() as conn:
            conn.execute(text("SET IDENTITY_INSERT Products ON"))
            batch.to_sql("Products", conn, if_exists="append", index=False)
            conn.execute(text("SET IDENTITY_INSERT Products OFF"))
        total += len(batch)
        if total % 50_000 == 0 or total == len(new_products):
            print(f"    ... {total:,} / {len(new_products):,}")
    print(f"  Products inserted: {total:,}")

In [ ]:
def insert_users(engine, df_users: pd.DataFrame):
    """Insert users, skipping any IDs that already exist."""
    insp = inspect(engine)
    if not insp.has_table("AspNetUsers"):
        print("  WARNING: AspNetUsers table does not exist. Skipping user insert.")
        return

    with engine.connect() as conn:
        existing = pd.read_sql("SELECT Id FROM AspNetUsers", conn)
    existing_ids = set(existing["Id"].values)

    new_users = df_users[~df_users["Id"].isin(existing_ids)]
    if len(new_users) == 0:
        print("  No new users to insert (all IDs already exist).")
        return

    print(f"  Inserting {len(new_users):,} new users ...")
    total = 0
    for start in range(0, len(new_users), BATCH_SIZE):
        batch = new_users.iloc[start : start + BATCH_SIZE]
        batch.to_sql("AspNetUsers", engine, if_exists="append", index=False)
        total += len(batch)
        if total % 50_000 == 0 or total == len(new_users):
            print(f"    ... {total:,} / {len(new_users):,}")
    print(f"  Users inserted: {total:,}")

In [ ]:
def insert_interactions(engine, df_interactions: pd.DataFrame):
    """Insert interactions into UserInteractions table."""
    create_user_interactions_table(engine)

    # Check if table already has data
    with engine.connect() as conn:
        count = conn.execute(text("SELECT COUNT(*) FROM UserInteractions")).scalar()
    if count > 0:
        print(f"  UserInteractions already has {count:,} rows. Skipping insert.")
        print("  (Drop the table manually if you want to re-import)")
        return

    print(f"  Inserting {len(df_interactions):,} interactions ...")
    total = 0
    for start in range(0, len(df_interactions), BATCH_SIZE):
        batch = df_interactions.iloc[start : start + BATCH_SIZE]
        batch.to_sql("UserInteractions", engine, if_exists="append", index=False)
        total += len(batch)
        if total % 100_000 == 0 or total == len(df_interactions):
            print(f"    ... {total:,} / {len(df_interactions):,}")
    print(f"  Interactions inserted: {total:,}")

---------------------------------------------------------------------------
Verification
---------------------------------------------------------------------------

In [ ]:
def verify_db():
    """Print summary counts from the database."""
    engine = get_engine()
    with engine.connect() as conn:
        for table in ["Products", "AspNetUsers", "UserInteractions"]:
            try:
                count = conn.execute(text(f"SELECT COUNT(*) FROM [{table}]")).scalar()
                print(f"  {table}: {count:,} rows")
            except Exception as e:
                print(f"  {table}: ERROR - {e}")

---------------------------------------------------------------------------
Main
---------------------------------------------------------------------------

In [ ]:
def main():
    # Replaced argparse block for Jupyter execution
    verify_mode = False
    dry_run_mode = False
    global MAX_TRANSACTIONS
    if MAX_TRANSACTIONS is None:
        MAX_TRANSACTIONS = 100000  # Default to 100k limit in notebook for speed
    print("=" * 60)
    print("Outfitly - H&M Data Preparation")
    print("=" * 60)

    if verify_mode:
        print("\n[Verify Mode] Checking database tables ...")
        verify_db()
        return

    # ---- Load parquet data ----
    print("\n[1/4] Loading parquet files ...")
    df_articles = load_parquet("articles.parquet")
    df_customers = load_parquet("customers.parquet")
    df_transactions = load_parquet("transactions_train.parquet")

    # ---- Prepare DataFrames ----
    print("\n[2/4] Preparing DataFrames ...")
    df_products = prepare_products(df_articles)
    df_users = prepare_users(df_customers)
    df_interactions = prepare_interactions(df_transactions)

    # ---- Export style tags ----
    print("\n[3/4] Exporting style tags ...")
    export_style_tags(df_articles)

    # ---- Preview ----
    print("\n--- Preview: Products ---")
    print(df_products.head(3).to_string(index=False))
    print(f"\n--- Preview: Users ---")
    print(df_users[["Id", "UserName", "Email"]].head(3).to_string(index=False))
    print(f"\n--- Preview: Interactions ---")
    print(df_interactions.head(3).to_string(index=False))

    if dry_run_mode:
        print("\n[DRY RUN] No data written to SQL Server.")
        print(f"  Products:     {len(df_products):,}")
        print(f"  Users:        {len(df_users):,}")
        print(f"  Interactions: {len(df_interactions):,}")
        return

    # ---- Insert into SQL Server ----
    print("\n[4/4] Inserting into SQL Server ...")
    engine = get_engine()

    print("\n  >> Products ...")
    insert_products(engine, df_products)

    print("\n  >> Users ...")
    insert_users(engine, df_users)

    print("\n  >> Interactions ...")
    insert_interactions(engine, df_interactions)

    print("\n" + "=" * 60)
    print("Data preparation complete!")
    print("=" * 60)

    # Final verification
    print("\n[Verification]")
    verify_db()

In [ ]:
if __name__ == "__main__":
    main()